In [ ]:
# Setup variables
BRANCH = "main"
import os
os.environ['MLFLOW_TRACKING_URI'] = "http://100.101.196.27:5000"
os.environ['COLAB_GPU'] = "True"


In [ ]:
# Mount google drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Clone repo
if not os.path.exists("/content/inat-phenology-cv"):
    !git clone -b {BRANCH} https://github.com/etiennegodin/inat-phenology-cv.git /content/inat-phenology-cv
%cd /content/inat-phenology-cv
!git pull origin {BRANCH} -q
!git restore .

Cloning into '/content/inat-phenology-cv'...
remote: Enumerating objects: 1318, done.
remote: Counting objects: 100% (473/473), done.
remote: Compressing objects: 100% (199/199), done.
remote: Total 1318 (delta 297), reused 390 (delta 222), pack-reused 845 (from 1)
Receiving objects: 100% (1318/1318), 208.94 KiB | 5.22 MiB/s, done.
Resolving deltas: 100% (822/822), done.
/content/inat-phenology-cv


In [ ]:
%%capture
# Instal project requirements
!pip install -q --upgrade pip
!pip install -e . -q


In [ ]:
%%capture
#Setup colab network to mlflow server
!curl -fsSL https://tailscale.com/install.sh | sh
!pip install -q "requests[socks]"

In [ ]:
%%bash
sudo setsid nohup bash -c '
while true; do
  tailscaled \
    --tun=userspace-networking \
    --socks5-server=localhost:1055 \
    --state=/tmp/tailscale.state \
    >> /tmp/tailscaled.log 2>&1
  echo "[$(date)] tailscaled exited, restarting in 2s" >> /tmp/tailscaled.log
  sleep 2
done
' < /dev/null > /tmp/tailscaled_supervisor.log 2>&1 &
echo "supervisor launched"

supervisor launched


In [ ]:
import subprocess
from google.colab import userdata
ts_auth_key = userdata.get("TAILSCALE_AUTH_KEY")
assert ts_auth_key
subprocess.run(
    [
        "sudo",
        "tailscale",
        "up",
        "--auth-key",
        ts_auth_key,
    ],
    check=True,
)
del ts_auth_key

In [ ]:
!sudo tailscale status

100.105.19.125  eab3eaef8183                       manateetiti@  linux    -                          
100.101.196.27  etienne-lenovo-ideapad-flex-15iml  manateetiti@  linux    -                          
100.112.113.49  pixel-7-pro                        manateetiti@  android  offline, last seen 3d ago  


In [ ]:
import requests

mlflow_proxies = {
    "http": "socks5h://localhost:1055",
    "https": "socks5h://localhost:1055",
}

r = requests.get(
    "http://100.101.196.27:5000/version",
    proxies=mlflow_proxies,
    timeout=5,
)

print(r.status_code)
print(r.text)

200
3.15.1


In [ ]:
# Copying mlflow.db from drive
os.makedirs("/content/data", exist_ok=True)
!cp -r "/content/drive/MyDrive/inat-phenology-cv/data/cv_raw.duckdb" "/content/data/"

In [ ]:
print("Copying images to local disk...")
if not os.path.exists("/content/images"):
  os.makedirs("/content/images", exist_ok=True)
  !tar -xf "/content/drive/MyDrive/inat-phenology-cv/data/images.tar.gz" -C /content/
  os.environ["INAT_IMAGE_DIR"] = "/content/images"

Copying images to local disk...


In [ ]:
!nvidia-smi

Fri Aug 21 16:11:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Gated attention, no dropout
!torch_pipe train -n 30 -p 3 --backbone bioclip --gated --attention_dropout 0.0 --seed 42 -name 'cv_inat_v0.8'

In [ ]:
from google.colab import runtime
runtime.unassign()